# LOMO 模型集成预测

使用 31 个 LOMO 二阶段最佳模型批量预测指定分子列表。每个模型以其 checkpoint 内保存的标准化参数处理输入；将反标准化后的预测值逐样本求平均，得到集成预测。

In [ ]:
import os
import re
import warnings
from glob import glob

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from kan import MultKAN
from sklearn.metrics import r2_score
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.nn.norm import BatchNorm

warnings.filterwarnings('ignore', message="Unable to accurately infer 'num_nodes'", category=UserWarning, module='torch_geometric.data.storage')

# 在此填写要预测的分子名列表，名称必须与下方 AVAILABLE_MOLECULE_NAMES 中的名称完全一致。
# MOLECULE_NAMES = ['A-BPPA', 'A-DDPPA', 'A-DPPA', 'A-HPPA', 'A-OPPA'
#                   , 'B-EPMBA', 'B-EPMEA', 'B-EPMHA', 'B-EPMOA', 'C-EEHHOA'
#                   , 'C-EETDOA', 'C-EHTDAA', 'C-HHOA', 'C-HTAA', 'C-TDOA', 'D-23EHPhOA'
#                   , 'D-246EHPhOA', 'D-24EHPhOA', 'D-25EHPhOA', 'D-26EHPhOA', 'D-mEHPhOA', 'D-oEHPhOA', 'D-pEHPhOA']
MOLECULE_NAMES = []

from pathlib import Path
# Run from the repository root or this notebook's directory.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'environment.yml').is_file() else Path.cwd().parent
MODEL_BASE_DIR = PROJECT_ROOT / '04_cross_validation' / 'lomo_results'
DATASET_PATH = PROJECT_ROOT / '02_datasets' / 'kare_gat_dataset.pt'
OUTPUT_DIR = PROJECT_ROOT / '05_prediction' / 'ensemble_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')
print(f'输出目录: {OUTPUT_DIR}')

In [ ]:
class GAT_KAN(nn.Module):
    def __init__(self, gat_hyper_params, kan_hyper_params, kan_save_act=False, device='cuda'):
        super().__init__()
        self.gat_layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        in_channels = 2

        for out_channels in gat_hyper_params['hidden_dims']:
            self.gat_layers.append(GATConv(in_channels, out_channels, heads=gat_hyper_params['heads']))
            self.batch_norms.append(BatchNorm(out_channels * gat_hyper_params['heads']))
            in_channels = out_channels * gat_hyper_params['heads']

        self.gat_out = gat_hyper_params['out_dim']
        self.gat_layers.append(GATConv(in_channels, self.gat_out, heads=1))
        self.batch_norms.append(BatchNorm(self.gat_out))

        kan_width = [self.gat_out + 7]
        for dim in kan_hyper_params['hidden_dims']:
            kan_width.append(dim if isinstance(dim, list) and len(dim) == 2 else [dim, 0])
        kan_width.append(1)

        kan_mult_arity = [tuple()]
        kan_mult_arity.extend(kan_hyper_params['mult_arity'])
        kan_mult_arity.append(tuple())
        self.kan = MultKAN(width=kan_width, mult_arity=kan_mult_arity, k=kan_hyper_params['k'], grid=kan_hyper_params['grid'], seed=666, save_act=kan_save_act, device=device)

    def forward(self, x, edge_index, batch, extra_features):
        for gat_layer, bn_layer in zip(self.gat_layers[:-1], self.batch_norms[:-1]):
            x = F.dropout(F.elu(bn_layer(gat_layer(x, edge_index))), p=0.5, training=self.training)
        x = self.batch_norms[-1](self.gat_layers[-1](x, edge_index))
        combined = torch.cat([global_mean_pool(x, batch), extra_features], dim=1)
        return self.kan(combined), combined


def fold_number(path):
    return int(re.search(r'fold_(\d+)_', os.path.basename(path)).group(1))


def load_torch(path, map_location=device):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


ELEMENT_SYMBOLS = {
    39: 'Y', 57: 'La', 58: 'Ce', 59: 'Pr', 60: 'Nd', 61: 'Pm', 62: 'Sm',
    63: 'Eu', 64: 'Gd', 65: 'Tb', 66: 'Dy', 67: 'Ho', 68: 'Er', 69: 'Tm',
    70: 'Yb', 71: 'Lu',
}
CONDITION_COLUMN_NAMES = [
    'Saponification (%)',
    'Extractant Concentration (mol/L)',
    'Solvent MPI (eV)',
    'Rare Earth Atomic Number',
    'RE3+ Radius',
    'RE3+ Electron Affinity (Hartree)',
    'Rare Earth Concentration (mol/L)',
]
# 用于区分曲线图的外部实验条件。第 4 项是稀土原子序号；第 5、6 项随稀土元素变化，
# 与横坐标元素绑定，因此不作为实验条件分组键。
EXPERIMENT_CONDITION_GROUP_INDICES = [0, 1, 2, 6]


def standardize_dataset(dataset, scalers):
    standardized = []
    for data in dataset:
        item = data.clone()
        item.atom_types_chg_scaled = (item.atom_types_chg.float() - scalers['atom_mean'].detach().cpu()) / scalers['atom_std'].detach().cpu()
        item.experiment_condition_tensor_scaled = (item.experiment_condition_tensor.float() - scalers['condition_mean'].detach().cpu()) / scalers['condition_std'].detach().cpu()
        standardized.append(item)
    return standardized


def load_model(checkpoint):
    model = GAT_KAN(checkpoint['gat_hyper_params'], checkpoint['kan_hyper_params'], kan_save_act=False, device=device).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model


def predict_with_model(model, checkpoint, dataset):
    batch = Batch.from_data_list(standardize_dataset(dataset, checkpoint['scalers'])).to(device)
    with torch.no_grad():
        prediction_scaled, combined = model(batch.atom_types_chg_scaled, batch.edge_index, batch.batch, batch.experiment_condition_tensor_scaled)
        predictions = prediction_scaled * checkpoint['scalers']['target_std'].to(device) + checkpoint['scalers']['target_mean'].to(device)
        gat_outputs = combined[:, :model.gat_out]
    return predictions.detach().cpu().numpy().reshape(-1), gat_outputs.detach().cpu().numpy()


def sort_conformer_ids(conformer_ids):
    return sorted(conformer_ids, key=lambda value: int(value) if str(value).isdigit() else str(value))


def condition_values_without_element(data):
    condition = data.experiment_condition_tensor.detach().cpu().numpy().reshape(-1)
    return [float(condition[index]) for index in EXPERIMENT_CONDITION_GROUP_INDICES]


def condition_key(values, decimals=10):
    return tuple(round(float(value), decimals) for value in values)


def sanitize_filename(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')


def format_condition_label(row):
    label_parts = []
    for index in EXPERIMENT_CONDITION_GROUP_INDICES:
        column = CONDITION_COLUMN_NAMES[index]
        value = row[column]
        label_parts.append(f'{column}: {value:g}')
    return '; '.join(label_parts)

In [ ]:
# 原始样本保留在 CPU；每折先用各自 scaler 标准化，随后再送到 GPU 推理。
allset = load_torch(DATASET_PATH, map_location='cpu')
AVAILABLE_MOLECULE_NAMES = sorted({str(data.name) for data in allset})
if not MOLECULE_NAMES:
    MOLECULE_NAMES = AVAILABLE_MOLECULE_NAMES.copy()
    print(f'MOLECULE_NAMES 为空，将预测全部 {len(MOLECULE_NAMES)} 个分子。')
missing_names = sorted(set(MOLECULE_NAMES) - set(AVAILABLE_MOLECULE_NAMES))
if missing_names:
    raise ValueError(f'以下分子名不存在: {missing_names}。可选名称: {AVAILABLE_MOLECULE_NAMES}')

fold_dirs = sorted(glob(os.path.join(MODEL_BASE_DIR, 'fold_*')), key=fold_number)
model_paths = []
for fold_dir in fold_dirs:
    fold = fold_number(fold_dir)
    matches = glob(os.path.join(fold_dir, f'best_model_fold{fold}-500-550-best_epoch*.pt'))
    if len(matches) != 1:
        raise FileNotFoundError(f'第 {fold} 折二阶段模型不唯一或缺失: {fold_dir}')
    model_paths.append(matches[0])
if not model_paths:
    raise FileNotFoundError(f'未找到模型: {MODEL_BASE_DIR}')

# 31 个模型只加载一次，随后对 MOLECULE_NAMES 中的每个分子重复使用。
ensemble_members = []
gat_out_dims = []
for index, model_path in enumerate(model_paths, start=1):
    checkpoint = load_torch(model_path)
    model = load_model(checkpoint)
    ensemble_members.append((model, checkpoint))
    gat_out_dims.append(model.gat_out)
    print(f'已加载模型 {index}/{len(model_paths)}: {os.path.basename(model_path)} | gat_out: {model.gat_out}')

if len(set(gat_out_dims)) != 1:
    raise ValueError(f'不同集成模型的 gat_out 维度不一致，无法求平均: {gat_out_dims}')
gat_out_dim = gat_out_dims[0]
lgc_column_names = [f'LGC{index}' for index in range(1, gat_out_dim + 1)]
gat_column_names = [f'GAT_{index}' for index in range(1, gat_out_dim + 1)]
mean_lgc_column_names = [f'Mean {column}' for column in lgc_column_names]

results_by_molecule = {}
all_prediction_frames = []
summary_records = []
for molecule_name in MOLECULE_NAMES:
    selected_data = [data for data in allset if str(data.name) == molecule_name]
    model_outputs = [predict_with_model(model, checkpoint, selected_data) for model, checkpoint in ensemble_members]
    all_model_predictions = [predictions for predictions, _ in model_outputs]
    all_model_lgc_outputs = [lgc_outputs for _, lgc_outputs in model_outputs]
    ensemble_prediction = np.mean(np.vstack(all_model_predictions), axis=0)
    ensemble_lgc_outputs = np.mean(np.stack(all_model_lgc_outputs, axis=0), axis=0)
    if ensemble_lgc_outputs.shape != (len(selected_data), gat_out_dim):
        raise ValueError(
            f'{molecule_name} 的 LGC 输出形状异常: {ensemble_lgc_outputs.shape}，'
            f'期望 ({len(selected_data)}, {gat_out_dim})'
        )
    molecule_mean_lgc_output = ensemble_lgc_outputs.mean(axis=0)
    true_values = np.array([data.extraction_rate.detach().cpu().numpy().reshape(-1)[0] for data in selected_data])

    condition_values = [condition_values_without_element(data) for data in selected_data]
    condition_keys = [condition_key(values) for values in condition_values]
    condition_to_group = {key: group_index + 1 for group_index, key in enumerate(sorted(set(condition_keys)))}
    results_df = pd.DataFrame({
        'Molecule': molecule_name,
        'Condition Group': [condition_to_group[key] for key in condition_keys],
        'Condition Key': [str(key) for key in condition_keys],
        'Conformer ID': [str(data.conformer_id) for data in selected_data],
        'Element Atomic Number': [int(data.experiment_condition_tensor.reshape(-1)[3].item()) for data in selected_data],
        'True Value': true_values,
        'Ensemble Prediction': ensemble_prediction,
    })
    results_df[lgc_column_names] = ensemble_lgc_outputs
    for condition_index in EXPERIMENT_CONDITION_GROUP_INDICES:
        results_df[CONDITION_COLUMN_NAMES[condition_index]] = [values[EXPERIMENT_CONDITION_GROUP_INDICES.index(condition_index)] for values in condition_values]
    results_df['Element Symbol'] = results_df['Element Atomic Number'].map(ELEMENT_SYMBOLS).fillna(results_df['Element Atomic Number'].astype(str))
    results_df['Residual'] = results_df['Ensemble Prediction'] - results_df['True Value']
    results_df['_Conformer Order'] = results_df['Conformer ID'].map(lambda value: int(value) if value.isdigit() else value)
    results_df = results_df.sort_values(['Condition Group', '_Conformer Order', 'Element Atomic Number']).drop(columns='_Conformer Order').reset_index(drop=True)

    molecule_r2 = r2_score(true_values, ensemble_prediction)
    results_by_molecule[molecule_name] = results_df
    all_prediction_frames.append(results_df)

    summary_record = {
        'Molecule': molecule_name,
        'Samples': len(results_df),
        'Conformers': results_df['Conformer ID'].nunique(),
        'Condition Groups': results_df['Condition Group'].nunique(),
        'Ensemble Models': len(ensemble_members),
        'GAT Output Dimension': gat_out_dim,
        'R2': molecule_r2,
    }
    summary_record.update({column: value for column, value in zip(gat_column_names, molecule_mean_lgc_output)})
    summary_record.update({column: value for column, value in zip(mean_lgc_column_names, molecule_mean_lgc_output)})
    summary_records.append(summary_record)
    print(f'完成分子: {molecule_name} | 样本数: {len(results_df)} | 构象数: {results_df["Conformer ID"].nunique()} | 实验条件组数: {results_df["Condition Group"].nunique()} | R²: {molecule_r2:.4f}')

all_predictions_df = pd.concat(all_prediction_frames, ignore_index=True)
summary_df = pd.DataFrame(summary_records)
all_results_excel_path = os.path.join(OUTPUT_DIR, 'LOMO_ensemble_all_prediction_results.xlsx')
with pd.ExcelWriter(all_results_excel_path) as writer:
    all_predictions_df.to_excel(writer, sheet_name='Predictions', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)

print(f'全部预测结果、逐样本 LGC 值和分子级平均 LGC 值已保存到: {all_results_excel_path}')
summary_df